# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `kimjonguk/my ai` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-2-2b-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `kimjonguk/my_to-v5` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0005 · steps 108 · seq 1024 · linear · 양자화 q4_k_m (데이터 54개)


In [ ]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


In [ ]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 (conversations Q&A)
내 지식이 **이 노트북에 직접 포함**돼 있어요 (업로드 불필요). 각 행 = `{conversations:[{user},{assistant}]}`


In [ ]:
import base64
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
# 내 지식(노트북에 포함) — base64로 안전하게 심어둠
_B64 = "..."
open("brain.jsonl", "w").write(base64.b64decode(_B64).decode("utf-8"))
ds = load_dataset("json", data_files="brain.jsonl", split="train")
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 108, learning_rate = 0.0005,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


In [ ]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 모델/버전마다 다름(<|turn> vs <start_of_turn>) → 실제 텍스트에서 자동 감지
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
_im = "<|turn>user\n" if "<|turn>user" in _t else "<start_of_turn>user\n"
_rm = "<|turn>model\n" if "<|turn>model" in _t else "<start_of_turn>model\n"
trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 학습 준비 완료")


In [ ]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [ ]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


## 💾 GGUF로 저장 (Connect AI 내장 엔진용)
테스트가 만족스러우면 업로드! (맨 앞에서 로그인했으니 바로 됩니다)


In [ ]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
print("메모리 정리 완료 — GGUF 변환 시작")
# 내 모델 = 장기 기억. q4_k_m GGUF 로 저장 + HF 업로드
model.push_to_hub_gguf("kimjonguk/my_to-v5", tokenizer, quantization_method="q4_k_m", token=True)
print("✅ 완료! Connect AI 앱 → 🤖 내 AI 팀 → HuggingFace에서 받기 → \"kimjonguk/my_to-v5\" 검색해서 내려받으면 내 모델로 바로 사용 (LM Studio 불필요)")
